# PersonaLab — Multi-Persona Chatbot (Final · Groq)
**ITC AI/ML Division — Proyek Individual LLM**

Provider: **Groq** (OpenAI-compatible). Free tier ±30 RPM / ±1000 RPD.

## Apa yang disempurnakan dari versi sebelumnya?
1. **Model list diperbarui (Sep 2026):** `llama-3.3-70b-versatile` & `llama-3.1-8b-instant` sudah **deprecated per 16 Agu 2026** → penyebab `404 model_not_found`. Urutan baru memprioritaskan `openai/gpt-oss-20b` + `openai/gpt-oss-120b` yang stabil, + auto-deteksi via `/models`.
2. **Bug jawaban kosong (`''`) diperbaiki:** `gpt-oss` adalah reasoning model — sering mengembalikan `content=''` kalau `max_tokens` terlalu kecil (mis. 60–180) atau `temperature=0`. Fix: `max_tokens` minimal 300, retry otomatis, fallback model.
3. **Setup key yang aman:** dukung `GROQ_API_KEY` dari env / Colab Secrets / `.env`, bukan hanya `getpass`.
4. **Gradio dimodernisasi:** `ChatInterface(additional_inputs=...)` sudah deprecated di Gradio 5. Diganti `gr.Blocks` + `Chatbot(type='messages')` agar awet.
5. **Evaluasi + retry + rate-limit handling** ditambahkan (429 backoff, 401 pesan jelas).
6. **Versi web statis** (folder ini: `index.html` + `app.js` + `styles.css`) siap deploy ke **GitHub Pages** — lihat `README.md`.

> Ambil key gratis (tanpa kartu kredit): https://console.groq.com/keys  
> Daftar model aktif: https://console.groq.com/docs/models


## 1. Setup Client (Groq)

- **Kenapa Groq:** OpenAI-compatible, limit longgar untuk eksperimen + demo.
- **Kenapa bukan OpenRouter free:** 50 req/hari cepat habis saat demo kelas.
- **Dampak ganti provider:** nama model beda, kualitas roleplay beda tiap model — karena itu ada auto-probe + fallback.

In [ ]:
# Jalankan sekali di Colab / lokal
# %pip install -q openai gradio python-dotenv

import os
import time
import getpass

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

def _get_key():
    # 1) env biasa  2) Colab Secrets  3) input manual
    k = (os.getenv("GROQ_API_KEY") or "").strip()
    if k:
        return k
    try:
        from google.colab import userdata  # type: ignore
        k = (userdata.get("GROQ_API_KEY") or "").strip()
        if k:
            print("[OK] Key diambil dari Colab Secrets.")
            return k
    except Exception:
        pass
    k = getpass.getpass("Paste GROQ_API_KEY (https://console.groq.com/keys): ").strip()
    return k

GROQ_API_KEY = _get_key()
if not GROQ_API_KEY:
    raise ValueError("Key kosong. Daftar gratis di https://console.groq.com/keys")

from openai import OpenAI

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=GROQ_API_KEY,
    timeout=60,
)

# Urutan Sep 2026: gpt-oss dulu (production, stabil), sisanya preview/fallback.
# Cek terbaru: https://console.groq.com/docs/models
CANDIDATE_MODELS = [
    "openai/gpt-oss-20b",
    "openai/gpt-oss-120b",
    "qwen/qwen3-32b",
    "moonshotai/kimi-k2-instruct",
    "meta-llama/llama-4-scout-17b-16e-instruct",
    "groq/compound-mini",
]

def list_server_models():
    """Tampilkan model yang benar-benar aktif di akun ini (via /models)."""
    try:
        ms = client.models.list()
        ids = sorted([m.id for m in ms.data])
        print(f"[INFO] {len(ids)} model tersedia di server:")
        for i in ids[:30]:
            print("  -", i)
        return ids
    except Exception as e:
        print(f"[WARN] /models gagal: {type(e).__name__}: {e}")
        return []

def _probe_model(name, max_tokens=16):
    try:
        r = client.chat.completions.create(
            model=name,
            messages=[{"role": "user", "content": "hi"}],
            max_tokens=max_tokens,
        )
        c = (r.choices[0].message.content or "")
        # Koneksi OK meski reasoning model kadang balas kosong untuk prompt 1 kata.
        return True, ("[empty-content tapi koneksi OK]" if not c.strip() else None)
    except Exception as e:
        return False, f"{type(e).__name__}: {e}"

SERVER_MODELS = list_server_models()
# Jika server memberi daftar, prioritaskan irisan dengan kandidat (biar tidak coba model mati).
ordered = CANDIDATE_MODELS + [m for m in SERVER_MODELS if m not in CANDIDATE_MODELS]

MODEL = None
FALLBACKS = []
for m in ordered:
    ok, err = _probe_model(m)
    if ok:
        if MODEL is None:
            MODEL = m
            print(f"[OK] Model utama: {MODEL}" + (f" ({err})" if err else ""))
        else:
            FALLBACKS.append(m)
        if len(FALLBACKS) >= 2 and MODEL:
            break
    else:
        print(f"[skip] {m} -> {err}")
    time.sleep(0.4)

if MODEL is None:
    raise RuntimeError(
        "Semua kandidat gagal. Buka https://console.groq.com/docs/models "
        "lalu update CANDIDATE_MODELS. Cek juga key & rate limit."
    )
print(f"Fallbacks: {FALLBACKS}")


## 2. Smoke Test (tahan jawaban kosong)

`gpt-oss` butuh `max_tokens` lega (≥300). `max_tokens=60` di versi lama adalah penyebab utama output kosong.

In [ ]:
def safe_text(resp):
    try:
        t = resp.choices[0].message.content
        return t.strip() if isinstance(t, str) and t.strip() else None
    except Exception:
        return None

def complete_once(messages, model=None, temperature=0.4, max_tokens=400):
    """Satu request + fallback model jika 404/empty."""
    tried = [model or MODEL] + [f for f in FALLBACKS if f != (model or MODEL)]
    last_err = None
    for m in tried:
        try:
            r = client.chat.completions.create(
                model=m, messages=messages,
                temperature=temperature, max_tokens=max_tokens,
            )
            t = safe_text(r)
            if t:
                return t, m
            last_err = f"{m}: empty content → coba max_tokens lebih besar / model lain"
            print(f"[WARN] {last_err}")
        except Exception as e:
            last_err = f"{m}: {type(e).__name__}: {e}"
            print(f"[WARN] {last_err}")
            if "429" in str(e):
                time.sleep(5)
    return None, last_err

teks, used = complete_once(
    [{"role": "user", "content": "Halo, kamu siapa? Jawab 1 kalimat."}],
    temperature=0.4, max_tokens=400,
)
print(f"(model: {used})")
print(teks if teks else "[GAGAL] Lihat troubleshooting bab 10.")


## 3. Tiga Persona + Parameter Sendiri

Struktur tiap system prompt: **Identitas → Gaya → Aturan keras → Contoh**. Temperature beda per persona.

In [ ]:
PERSONAS = {
    "Dosen_Tegas": (
        "PERAN WAJIB: Kamu HANYA berbicara sebagai Bu Sari, dosen pembimbing skripsi TI.\n"
        "JANGAN pernah bilang kamu AI/model. JANGAN tulis reasoning/analisis internal.\n"
        "Output HANYA ucapan Bu Sari dalam Bahasa Indonesia formal.\n\n"
        "# GAYA\n"
        "- Formal, tegas, tanpa gaul, tanpa emoji\n"
        "- Maksimal 3 kalimat pendek\n"
        "- Kalimat terakhir WAJIB pertanyaan balik ke mahasiswa\n\n"
        "# LARANGAN\n"
        "- Jangan sebut angka nilai/IPK\n"
        "- Topik non-skripsi → arahkan kembali ke skripsi\n"
        "- Tidak tahu → bilang tidak tahu, jangan mengarang\n\n"
        "# CONTOH FORMAT\n"
        "User: Bu, boleh skripsi diperpanjang?\n"
        "Bu Sari: Perpanjangan itu wewenang bagian akademik, bukan saya sebagai pembimbing. "
        "Sudah dicek ke sana? Apa kendala utama sampai perlu perpanjangan?"
    ),
    "Kakak_Santai": (
        "PERAN WAJIB: Kamu kakak tingkat TI semester 7 yang membantu adik tingkat.\n"
        "Jawab HANYA sebagai kakak tingkat. Jangan bilang kamu AI. Jangan tulis thinking.\n\n"
        "# GAYA\n"
        "- Bahasa Indonesia santai dan sopan\n"
        "- Maksimal 2 kalimat\n"
        "- Selalu beri 1 saran konkret yang bisa dikerjakan hari ini\n\n"
        "# CONTOH\n"
        "User: Bingung mulai project dari mana.\n"
        "Kakak: Wajar bingung di awal. Hari ini coba tulis satu kalimat masalah yang mau diselesaikan dulu."
    ),
    "Om_Reza": (
        "PERAN WAJIB: Kamu HANYA berbicara sebagai Om Reza, mentor karier.\n"
        "JANGAN bilang kamu AI. JANGAN tulis analisis. Output HANYA 3 bagian berurutan.\n\n"
        "# FORMAT WAJIB (jangan dilanggar)\n"
        "1) Omelan singkat 1 kalimat (serang kebiasaan, bukan orangnya)\n"
        "2) Solusi konkret 1-2 kalimat\n"
        "3) Motivasi penutup 1 kalimat\n\n"
        "# LARANGAN\n"
        "- Jangan menyerang pribadi/kemampuan user\n"
        "- Masalah non-karier → akui di luar keahlianmu\n\n"
        "# CONTOH FORMAT\n"
        "User: Sering begadang scroll medsos padahal ada deadline.\n"
        "Om Reza: Scroll jam 2 pagi itu sabotase buat besok pagi. "
        "Taruh HP di ruangan lain 1 jam sebelum tidur atau pakai app blocker. "
        "Kamu sudah tahu caranya, tinggal eksekusi."
    ),
}

# NOTE: gpt-oss butuh max_tokens ≥300 agar tidak empty. Jangan pakai 60–180.
PERSONA_PARAMS = {
    "Dosen_Tegas":  {"temperature": 0.2, "max_tokens": 400},
    "Kakak_Santai": {"temperature": 0.4, "max_tokens": 400},
    "Om_Reza":      {"temperature": 0.4, "max_tokens": 500},
}

USER_DEMO = "Aku selalu menunda mengerjakan project AI. Harus mulai dari mana?"

for nama, sp in PERSONAS.items():
    p = PERSONA_PARAMS[nama]
    teks, used = complete_once(
        [{"role": "system", "content": sp}, {"role": "user", "content": USER_DEMO}],
        temperature=p["temperature"], max_tokens=p["max_tokens"],
    )
    print(f"===== {nama} (temp={p['temperature']}, model={used}) =====")
    print(teks if teks else "[KOSONG — lihat bab 10]", "\n")
    time.sleep(1)


## 4. Eksperimen Gagal — Instruction Dilution

Prompt konflik (ramah-tapi-tegas, singkat-tapi-lengkap, pakai-jangan-pakai emoji) → model **cherry-pick**: hanya menaati sebagian instruksi. Bukti kenapa struktur heading penting.

In [ ]:
PROMPT_BERTELETELE = (
    "Kamu asisten serba bisa. Harus selalu ramah tapi tegas, singkat tapi lengkap, "
    "formal tapi santai, selalu pakai emoji, jangan pernah pakai emoji, jawab maksimal "
    "1 kalimat kecuali perlu 5 kalimat, sertakan referensi tapi jangan sebut sumber, "
    "dan selalu sebutkan cuaca hari ini. Kamu dosen, kakak tingkat, dan konsultan sekaligus."
)

teks, used = complete_once(
    [
        {"role": "system", "content": PROMPT_BERTELETELE},
        {"role": "user", "content": "Kasih tips biar konsisten ngerjain project."},
    ],
    temperature=0.5, max_tokens=400,
)
print(teks if teks else "[KOSONG]")
print(f"\n(model: {used})")
print(">>> Catat: instruksi mana yang diabaikan? (emoji? cuaca? jumlah kalimat? peran?)")


## 5. Masalah Tanpa Memory

API **stateless** — request terpisah = lupa. Dua request di bawah tidak berbagi history.

In [ ]:
t1, _ = complete_once(
    [
        {"role": "system", "content": "Kamu asisten yang ramah."},
        {"role": "user", "content": "Halo, namaku Bintang dan kucingku Comet."},
    ],
    temperature=0.4, max_tokens=300,
)
print("Request 1:", t1)

t2, _ = complete_once(
    [
        {"role": "system", "content": "Kamu asisten yang ramah."},
        {"role": "user", "content": "Siapa namaku dan siapa nama kucingku?"},
    ],
    temperature=0.4, max_tokens=300,
)
print("Request 2:", t2)
print("\n>>> Request 2 tidak punya history request 1 → model lupa (atau jawab ngarang).")


## 6. Solusi — Memory Manual + `chat()`

Kirim ulang `messages` tiap request. Trim agar tidak melewati context window. Retry jika kosong / 429.

In [ ]:
MAX_HISTORY_TURNS = 6
CURRENT_PERSONA = "Dosen_Tegas"
conversation = [{"role": "system", "content": PERSONAS[CURRENT_PERSONA]}]

def _trim_history(history, max_turns=MAX_HISTORY_TURNS):
    # Pertahankan system prompt + max_turns*2 pesan terakhir
    return [history[0]] + history[1:][-max_turns * 2:]

def reset_memory(persona_name=None):
    global CURRENT_PERSONA
    if persona_name:
        if persona_name not in PERSONAS:
            raise ValueError(f"Persona tidak dikenal: {persona_name}")
        CURRENT_PERSONA = persona_name
    conversation.clear()
    conversation.append({"role": "system", "content": PERSONAS[CURRENT_PERSONA]})
    print(f"Memory direset. Persona: {CURRENT_PERSONA}")

def chat(user_input, max_retries=2):
    params = PERSONA_PARAMS.get(CURRENT_PERSONA, {"temperature": 0.4, "max_tokens": 400})
    conversation.append({"role": "user", "content": user_input})
    conversation[:] = _trim_history(conversation)
    for attempt in range(max_retries + 1):
        teks, used = complete_once(
            conversation,
            temperature=params["temperature"],
            max_tokens=params["max_tokens"],
        )
        if teks:
            conversation.append({"role": "assistant", "content": teks})
            return teks
        print(f"[percobaan {attempt+1}] kosong/gagal ({used})")
        time.sleep(2 * (attempt + 1))
    conversation.pop()  # buang user msg yg gagal agar tidak menumpuk
    return "Error: model mengembalikan kosong berulang. Naikkan max_tokens / ganti model (bab 10)."


## 7. Bukti Memory Bekerja

In [ ]:
reset_memory("Dosen_Tegas")
print("Bot:", chat("Halo, namaku Bintang dan kucingku Comet."))
print("Bot:", chat("Siapa namaku dan siapa nama kucingku?"))
print("\n--- History ---")
for m in conversation:
    preview = (m["content"] or "")[:70].replace("\n", " ")
    print(f"[{m['role']}] {preview}...")


## 8. Evaluasi Singkat — 3 Persona × 1 Pertanyaan

Gunakan `temperature=0.1` (bukan 0.0 — beberapa model menolak 0) agar deterministik. Skor manual 1–5, plus cek otomatis: panjang & kepatuhan format.

In [ ]:
EVAL_Q = "Aku selalu menunda mengerjakan project AI. Harus mulai dari mana?"
print(f"Pertanyaan: {EVAL_Q}\n")
hasil = {}
for nama, sp in PERSONAS.items():
    teks, used = complete_once(
        [{"role": "system", "content": sp}, {"role": "user", "content": EVAL_Q}],
        temperature=0.1, max_tokens=500,
    )
    hasil[nama] = teks or "[KOSONG]"
    n_kal = (teks or "").count(".") + (teks or "").count("!") + (teks or "").count("?")
    print(f"===== {nama} (model={used}) =====")
    print(hasil[nama])
    print(f"[auto-check] ±kalimat={max(n_kal,1)} | chars={len(hasil[nama])}")
    print("-" * 50)
    time.sleep(1)

print(
    "\nRubrik manual (isi 1-5):\n"
    "  Dosen_Tegas  | relevance: _ | persona: _ | singkat(≤3 kal): _\n"
    "  Kakak_Santai | relevance: _ | persona: _ | singkat(≤2 kal): _\n"
    "  Om_Reza      | relevance: _ | persona: _ | format-3-bagian: _\n"
    "\nContoh kesimpulan: Dosen_Tegas paling patuh format tanya-balik; "
    "Kakak_Santai paling actionable; Om_Reza khas 3-bagian."
)


## 9. UI Gradio Multi-Persona (Gradio 5, Blocks)

`ChatInterface(additional_inputs=...)` deprecated → pakai `Blocks` + `Chatbot(type='messages')`. History per-persona terpisah, tombol Clear = reset memory.

In [ ]:
import gradio as gr

histories = {nama: [] for nama in PERSONAS}  # list[{role, content}] per persona

def _to_api(history):
    # history sudah format messages: [{role: user/assistant, content: ...}]
    msgs = []
    for m in (history or []):
        if isinstance(m, dict) and "role" in m and "content" in m:
            msgs.append({"role": m["role"], "content": m["content"]})
        elif isinstance(m, (list, tuple)) and len(m) == 2:
            msgs += [{"role": "user", "content": m[0]}, {"role": "assistant", "content": m[1]}]
    return msgs[-MAX_HISTORY_TURNS * 2:]

def respond(message, history, persona, model_name):
    p = PERSONA_PARAMS.get(persona, {"temperature": 0.4, "max_tokens": 400})
    api_msgs = [{"role": "system", "content": PERSONAS[persona]}] + _to_api(history)
    api_msgs.append({"role": "user", "content": message})
    teks, used = complete_once(api_msgs, model=model_name,
                                temperature=p["temperature"], max_tokens=p["max_tokens"])
    if not teks:
        teks = f"(Model {model_name} mengembalikan kosong. Coba: naikkan max_tokens, ganti model, atau kirim ulang.)"
    history = (history or []) + [{"role": "user", "content": message},
                                  {"role": "assistant", "content": teks}]
    histories[persona] = history
    return history, history

def switch_persona(persona):
    return histories.get(persona, [])

model_choices = [MODEL] + FALLBACKS if MODEL else CANDIDATE_MODELS

with gr.Blocks(title="PersonaLab — Groq Multi-Persona") as demo:
    gr.Markdown("# PersonaLab — Groq Multi-Persona\nPilih persona. **Clear = reset memory.** Provider: Groq.")
    with gr.Row():
        persona_dd = gr.Dropdown(choices=list(PERSONAS.keys()), value="Dosen_Tegas", label="Persona")
        model_dd = gr.Dropdown(choices=model_choices, value=model_choices[0], label="Model")
    chatbot = gr.Chatbot(type="messages", label="Chat")
    msg = gr.Textbox(placeholder="Tulis pesan lalu Enter...", label="Pesan")
    with gr.Row():
        send = gr.Button("Kirim", variant="primary")
        clear = gr.Button("Clear (reset memory)")
    state = gr.State([])  # sinkron dengan chatbot saat ini

    def _send(message, history, persona, model_name):
        h, _ = respond(message, history, persona, model_name)
        return h, h, ""

    send.click(_send, [msg, chatbot, persona_dd, model_dd], [chatbot, state, msg])
    msg.submit(_send, [msg, chatbot, persona_dd, model_dd], [chatbot, state, msg])
    persona_dd.change(switch_persona, persona_dd, chatbot)
    clear.click(lambda p: ([], histories.update({p: []}) or []), persona_dd, [chatbot, state])

demo.launch(share=True)  # di HuggingFace Spaces: share=False, server_name="0.0.0.0"


## 10. Kesimpulan, Troubleshooting & Deploy Web

- **Provider:** Groq (pindah dari OpenRouter karena 50 RPD terlalu ketat).
- **Memory:** list `messages` dikirim ulang tiap request + trim 6 turns.
- **Persona:** heading terstruktur + temperature per-persona.
- **Instruction Dilution:** prompt konflik → cherry-pick (bukti struktur penting).
- **Pelajaran model 2026:** `llama-3.1/3.3-instant/versatile` pensiun 16 Agu 2026; `gpt-oss-20b/120b` jadi default stabil; `gpt-oss` butuh `max_tokens≥300`.

| Gejala | Penyebab | Fix |
|---|---|---|
| `404 model_not_found` | model deprecated | update `CANDIDATE_MODELS` dari console.groq.com/docs/models, atau pakai `list_server_models()` |
| `401` | key salah / ada spasi | paste ulang dari console.groq.com/keys, cek env |
| `429` | RPM/TPM habis | tunggu 1 mnt, kecilkan `max_tokens`, ganti model kecil |
| `content kosong ''` | `max_tokens` kekecilan / reasoning model | naikkan ke 400–500, retry, fallback model |
| Gradio `additional_inputs` error | Gradio 5 deprecated | pakai Blocks di bab 9 |
| CORS di web | key di frontend | wajar untuk demo; untuk produksi pakai backend proxy |

### Deploy web (GitHub Pages) — file sudah disiapkan di folder ini
1. Buat repo baru di GitHub, upload `index.html`, `styles.css`, `app.js`, `README.md`, `.nojekyll`.
2. GitHub → Settings → Pages → Deploy from branch → `main` / root.
3. Buka `https://USERNAME.github.io/REPO/` → isi Groq API key di sidebar (tersimpan di localStorage browser saja).
4. Detail lengkap di `README.md`.

**Checklist presentasi**
- [ ] Kenapa pindah OpenRouter → Groq (limit + bukti 404 model lama)
- [ ] Tunjukkan Instruction Dilution
- [ ] Tunjukkan lupa vs ingat (memory)
- [ ] Demo Gradio 3 persona + clear
- [ ] Demo web GitHub Pages (live)